In [ ]:
import org.springframework.ai.chat.client.ChatClient
import org.springframework.ai.chat.client.advisor.MessageChatMemoryAdvisor
import org.springframework.ai.chat.memory.InMemoryChatMemoryRepository
import org.springframework.ai.chat.memory.MessageWindowChatMemory
import org.springframework.ai.openai.OpenAiChatModel
import org.springframework.ai.openai.OpenAiChatOptions
import org.springframework.ai.openai.api.OpenAiApi
import org.springframework.ai.chat.client.entity
val apiKey = System.getenv("OPENAI_API_KEY") ?: "YOUR_OPENAI_API_KEY"

val openAiApi = OpenAiApi.builder().apiKey(apiKey).build()
val openAiChatOptions = OpenAiChatOptions.builder()
    .model(OpenAiApi.ChatModel.GPT_5_CHAT_LATEST)
    .temperature(0.8)
    .build()
val chatModel = OpenAiChatModel.builder().openAiApi(openAiApi).defaultOptions(openAiChatOptions).build()
val chatClient = ChatClient.builder(chatModel)
    .defaultAdvisors(
        MessageChatMemoryAdvisor.builder(MessageWindowChatMemory.builder().chatMemoryRepository(InMemoryChatMemoryRepository()).build()).build(),
    ).build()


### Domain

In [ ]:
data class UserProfile(
    val userType: String,
    val id: String,
    val description: String,
    val goals: List<String>
)

data class Scenario(
    val scenarioName: String,
    val description: String,
    val userProfileId: String,
    val userIntent: String,
    val needs: List<String>
)

data class ScenarioPrompt(
    val id: String,
    val question: String,
    val userType: String,
    val scenarioName: String
) {
    override fun toString(): String = """<tr><td><strong>Question</strong></td><td>$question</td></tr><tr><td><strong>User</strong></td><td style="text-align: left">$userType</td><tr><td><strong>Scenario</strong></td><td style="text-align: left">$scenarioName</td></tr>"""
}


# EDD: Eval Driven Development

## Step 1: (Synthetic) Data Generation
- **Goals**:
  - A chat assistant for KotlinConf attendees and speakers that helps them quickly find accurate, up-to-date conference facts and build a personal session plan.
- **Users**
  - First-time attendee
  - Kotlin frontend developer
  - Kotlin backend developer
- **Scenarios**


### Users

In [ ]:
val users = listOf(
    UserProfile(
        userType = "First-time attendee",
        id = "first_time_attendee",
        description = "New to KotlinConf (and often the venue). Asks broad, practical questions and prefers guided suggestions backed by exact facts.",
        goals = listOf(
            "Get oriented quickly (venue, schedule, logistics).",
            "Discover suitable sessions using plain-language queries.",
            "Create a small shortlist of sessions to attend."
        )
    ),
    UserProfile(
        userType = "Kotlin backend developer",
        id = "kotlin_backend_developer",
        description = "Technical attendee optimizing for relevance and depth for backend topics. Uses specific topic/level queries and compares session options.",
        goals = listOf(
            "Find highly relevant sessions by topic, technology, and skill level.",
            "Compare sessions based on content, speaker, time, and room.",
            "Build and refine a conflict-free personal schedule."
        )
    ),
    UserProfile(
        userType = "Kotlin frontend developer",
        id = "kotlin_frontend_developer",
        description = "Technical attendee looking for a frontend-focused session. Uses specific topic/level queries and compares session options.",
        goals = listOf(
            "Find highly relevant sessions by topic, technology, and skill level.",
            "Compare sessions based on content, speaker, time, and room.",
            "Build and refine a conflict-free personal schedule."
        )
    )
)

### Scenarios

In [ ]:
val scenarios = listOf(
    Scenario(
        scenarioName = "FirstTimeAttendee_OrientationAndPlan",
        description = "A first-time attendee wants to understand venue logistics and discover suitable beginner-friendly sessions to attend.",
        userProfileId = "first_time_attendee",
        userIntent = "Get oriented and decide what to attend",
        needs = listOf(
            "venue_info",
            "session_info",
            "semantic_session_search",
            "shortlist_or_booking"
        )
    ),
    Scenario(
        scenarioName = "Developer_TargetedSemanticSearchAndSchedule",
        description = "A Kotlin developer searches for highly relevant technical sessions, compares options, and builds a personal schedule.",
        userProfileId = "kotlin_developer",
        userIntent = "Optimize for technical relevance and depth",
        needs = listOf(
            "semantic_session_search",
            "session_info",
            "shortlist_or_booking"
        )
    ),
    Scenario(
        scenarioName = "Developer_CreateConferenceSchedule",
        description = "A Kotlin backend or frontend developer creates a personal conference schedule reflecting his/her preferences.",
        userProfileId = "kotlin_developer",
        userIntent = "Compose personalized conference schedule",
        needs = listOf(
            "session_info",
            "venue_info",
            "semantic_session_search",
            "session_preferences"
        )
    )
)

### Generate Test Prompts
Combine Scenarios and Users into specific, realistic prompts that can be used to evaluate the assistant's performance in each scenario. Each prompt should reflect the user's intent and needs as defined in the scenarios.

In [23]:
chatClient.prompt()
    .system { it.text("You are a helpful testdata generation assistant for an eval driven development workflow")
    }
    .user { it.text("""
        Given following user profiles:
        $users
        ---
        And following scenarios:
        $scenarios
        ---
        Generate two prompts for each user profile matching a different scenario.
        It must be phrased as realistic questions that a user of that type might ask the assistant. Each prompt should reflect the user's intent and needs as defined in the scenarios.
        """)
}.call().entity<List<ScenarioPrompt>>().joinToString("<hr>") { "<table>$it</table>" }.let{HTML(it)}


Question,"I'm new to KotlinConf—where do I go to pick up my badge, and which beginner sessions would be the best starting point for me?"
User,First-time attendee
Scenario,FirstTimeAttendee_OrientationAndPlan
Question,Can you show me how the venue is laid out and recommend a few introductory Kotlin sessions I could shortlist for my first day?
User,First-time attendee
Scenario,FirstTimeAttendee_OrientationAndPlan
Question,"Which Kotlin backend talks focus on microservices and performance optimization, and how can I organize them into a schedule without conflicts?"
User,Kotlin backend developer
Scenario,Developer_TargetedSemanticSearchAndSchedule
Question,Can you help me build a personal schedule that prioritizes advanced backend topics like coroutine management and server design?
User,Kotlin backend developer
